# Forecast robuste - Credit a l'equipement

Ce notebook est construit comme outil de prise de decision, pas comme generateur de prediction magique.

Principe:
- on reserve les 60 derniers jours comme hold-out final;
- la selection des modeles se fait avant le hold-out, via walk-forward rolling origin;
- les modeles complexes sont compares a des baselines simples;
- si le gain vs meilleur baseline est < 5%, le baseline est retenu;
- la projection 90 jours inclut un intervalle empirique et un niveau de fiabilite.


## Configuration

In [ ]:
from pathlib import Path

import pandas as pd

from robust_forecast_engine import ForecastConfig, load_workbook, run_bundle


config = ForecastConfig(
    input_path=Path("in/reserve_in.xlsx"),
    output_dir=Path("out"),
    holdout_len=60,
    forecast_horizon=90,
    wf_folds=5,
    wf_horizon=30,
    min_gain_vs_baseline_pct=5.0,
)

targets = ["Credit Décaissement_crédits à lequipement"]


## Inspection rapide du fichier source

In [ ]:
df = load_workbook(config)
print(df.shape)
print(df.index.min(), "->", df.index.max())
display(df[targets].describe().T)


## Execution de la pipeline robuste

In [ ]:
results = run_bundle(
    name="credit_equipement",
    targets=targets,
    config=config,
    add_total=False,
)


## Synthese decisionnelle

In [ ]:
summary_rows = []
for result in results:
    diag = result.diagnostics.set_index("metric")["value"]
    summary_rows.append({
        "serie": result.target,
        "modele_retenu": diag["final_model"],
        "modele_propose": diag["proposed_model"],
        "meilleur_baseline": diag["best_baseline"],
        "mape_holdout": diag["holdout_MAPE"],
        "gain_vs_baseline_pct": diag["gain_vs_best_baseline_pct"],
    })
summary = pd.DataFrame(summary_rows)
display(summary)


## Projection 90 jours

In [ ]:
for result in results:
    display(result.projection.head(20))


## Comparaison des modeles

In [ ]:
for result in results:
    cols = ["model", "holdout_MAPE", "wf_MAPE_median", "ensemble_weight"]
    display(result.candidates[cols].sort_values("holdout_MAPE"))
